In [1]:
import os
import json
from pathlib import Path

def load_jsons(base_dir):
    all_data = {}

    # Walk through all files under base_dir
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith(".json"):
                file_path = Path(root) / file

                # Extract folder name (last part of path)
                folder_name = Path(root).name

                # Take the second-to-last numeric block (before the last underscore + digit)
                # Example: "7099892__0" -> "7099892"
                if "__" in folder_name:
                    key = folder_name.split("__")[-2]
                else:
                    # fallback if structure is slightly different
                    key = folder_name

                try:
                    with open(file_path, "r") as f:
                        data = json.load(f)
                    all_data[key] = data
                except Exception as e:
                    print(f"Could not load {file_path}: {e}")

    return all_data


base_directory = "mastest/results/all"  # adjust if needed
combined = load_jsons(base_directory)

print(f"Loaded {len(combined)} JSON files")
# # Optional: save combined dict to one file
# with open("combined.json", "w") as out:
#     json.dump(combined, out, indent=2)


Loaded 39 JSON files


In [6]:
res = {}
for k, v in combined.items():
    res[k] = v["results"]

with open("combined.json", "r") as fin:
    evals = json.load(fin)

In [4]:
tokens = [1, 2, 4, 6] 

hp_10 = [68483, 68484, 68485, 68482]
pc_10 = [68465, 68466, 68467, 68464]
hp_30 = [7099892, 68487, 68488, 68486]
pc_30 = [7106630, 68469, 68470, 68468]
hp_50 = [68492, 7099896, 7099897, 7099895]
pc_50 = [68476, 7099901, 7105356, 7099899]
hp_30_ri = [None, 68490, 68491, 68489]
pc_30_ri = [68472, 68473, 68474, 68471]
hp_50_ta = [68494, 68495, 68496, 68493]
pc_50_ta = [68479, 68480, 68499, 68478]

In [7]:
# preprocess ids
def preprecess_eval(rawish_evals):
    res = {}
    for k, v in rawish_evals.items():
        tmp = {}
        tmp['arc_challenge'] = v["arc_challenge"]["acc,none"]
        tmp['arc_easy'] = v["arc_easy"]["acc,none"]
        tmp['hellaswag'] = v["hellaswag"]["acc,none"]
        tmp['lambada_openai'] = v["lambada_openai"]["acc,none"]
        tmp['openbookqa'] = v["openbookqa"]["acc,none"]
        tmp['piqa'] = v["piqa"]["acc,none"]
        tmp['sciq'] = v["sciq"]["acc,none"]
        tmp['social_iqa'] = v["social_iqa"]["acc,none"]
        tmp['winogrande'] = v["winogrande"]["acc,none"]
        res[k] = tmp
    return res
evals = preprecess_eval(evals)

In [26]:
from statistics import mean
import plotly.graph_objects as go
import math
from plotly.subplots import make_subplots


def create_combined_chart(tasks, tokens, pc_metric_vals, hp_metric_vals):
    # Calculate grid dimensions (3 columns, ceil(n_tasks/3) rows)
    n_tasks = len(tasks)
    n_cols = 3
    n_rows = math.ceil(n_tasks / n_cols)
    
    # Create subplots
    fig = make_subplots(
        rows=n_rows, 
        cols=n_cols,
        subplot_titles=tasks,
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )
    
    # Colors
    color_pc = "blue"
    color_hp = "red"
    
    for i, task in enumerate(tasks):
        row = (i // n_cols) + 1
        col = (i % n_cols) + 1
        
        pc_vals = pc_metric_vals[task]
        hp_vals = hp_metric_vals[task]
        
        # Filter out None values for PC
        pc_tokens_filtered = [t for t, v in zip(tokens, pc_vals) if v is not None]
        pc_vals_filtered = [v for v in pc_vals if v is not None]
        
        # Filter out None values for HP
        hp_tokens_filtered = [t for t, v in zip(tokens, hp_vals) if v is not None]
        hp_vals_filtered = [v for v in hp_vals if v is not None]
        
        # Add PC trace (only if there are valid values)
        if pc_vals_filtered:
            fig.add_trace(
                go.Scatter(
                    x=pc_tokens_filtered, 
                    y=pc_vals_filtered,
                    mode='lines+markers',
                    name='PC',
                    legendgroup="PC",
                    line=dict(color=color_pc, dash="solid"),
                    marker=dict(symbol="circle", color=color_pc),
                    showlegend=(i == 0)  # Only show legend for first trace
                ),
                row=row, col=col
            )
        
        # Add HP trace (only if there are valid values)
        if hp_vals_filtered:
            fig.add_trace(
                go.Scatter(
                    x=hp_tokens_filtered, 
                    y=hp_vals_filtered,
                    mode='lines+markers',
                    name='HP',
                    legendgroup="HP",
                    line=dict(color=color_hp, dash="solid"),
                    marker=dict(symbol="square", color=color_hp),
                    showlegend=(i == 0)  # Only show legend for first trace
                ),
                row=row, col=col
            )
        
        # Set y-axis range for this subplot (only use non-None values)
        all_vals_filtered = [v for v in pc_vals + hp_vals if v is not None]
        if all_vals_filtered:
            y_center = mean(all_vals_filtered)
            fig.update_yaxes(
                range=[y_center - 0.075, y_center + 0.075],
                title_text="Score" if row == n_rows else "",  # Only bottom row gets y-axis label
                row=row, col=col
            )
        
        # Set x-axis title for bottom row
        if row == n_rows:
            fig.update_xaxes(title_text="Tokens (Billion)", row=row, col=col)
    
    # Update overall layout
    fig.update_layout(
        title="PC vs HP Performance Comparison Across Tasks",
        height=300 * n_rows,  # Adjust height based on number of rows
        width=1200,
        legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.8)"),
        font=dict(size=10)
    )
    
    return fig

def chart_comparison(pc_ids, hp_ids, evals, tokens = [1,2,4,6], tasks=['arc_challenge', 'arc_easy', 'hellaswag', 'lambada_openai', 'openbookqa', 'piqa', 'sciq', 'social_iqa', 'winogrande'], save_path = "test.png"):
    def scrap_metrics(mids, evals, tasks):
        res = {k:[] for k in tasks}
        for mid in mids:
            me = evals[str(mid)]
            for t in tasks:
                res[t].append(me[t])
        return res

    mpc_metric_vals = scrap_metrics(pc_ids, evals, tasks)
    hp_metric_vals = scrap_metrics(hp_ids, evals, tasks)
    
    # print(m1_metric_vals)


    # Create and display the combined chart
    combined_fig = create_combined_chart(tasks, tokens, mpc_metric_vals, hp_metric_vals)

    n_rows = math.ceil(len(tasks) / 3)  # same as inside function
    size_c = 150
    size = 8

    combined_fig.write_image(
        save_path,
        format="png",
        width=size_c*size,   # increase width
        height=size_c * n_rows * 2,  # scale height
        scale=2       # improves DPI/sharpness
    )

    return combined_fig

comp_10 = chart_comparison(pc_10, hp_10, evals, save_path = "mastest/charts/final/comp_10.png")
comp_30 = chart_comparison(pc_30, hp_30, evals, save_path = "mastest/charts/final/comp_30.png")
comp_50 = chart_comparison(pc_50, hp_50, evals, save_path = "mastest/charts/final/comp_50.png")
comp_50_ta = chart_comparison(pc_50_ta, hp_50_ta, evals, save_path = "mastest/charts/final/comp_50_ta.png")
comp_30_ri = chart_comparison(pc_30_ri, hp_30_ri, evals, save_path = "mastest/charts/final/comp_30_ri.png")


KeyError: 'None'

In [9]:
print([
    "arc_challenge",
    "arc_easy",
    "hellaswag",
    "lambada_openai",
    "openbookqa",
    "piqa",
    "sciq",
    "social_iqa",
    "winogrande"
])

['arc_challenge', 'arc_easy', 'hellaswag', 'lambada_openai', 'openbookqa', 'piqa', 'sciq', 'social_iqa', 'winogrande']
